# Superstore Sales Analytics - Interview Presentation Notebook

This notebook is designed for portfolio and interview storytelling.

Structure:
1. Business problem
2. Data preparation
3. 8 strongest visuals
4. 5 key insights
5. 3 business recommendations
6. ML prediction summary
7. Executive conclusion

## 1) Problem Statement

Superstore wants to improve profitability while maintaining revenue growth.

Questions:
- Which areas drive sales and profit?
- Which areas create losses?
- What role does discount play in profit erosion?
- Can we predict order-level profit with ML?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid', palette='Set2')

## 2) Load and Prepare Data

In [ ]:
candidate_paths = [
    Path('SuperStoreOrders_SuperStoreOrders.csv'),
    Path('../SuperStoreOrders_SuperStoreOrders.csv'),
    Path('data/raw/SuperStoreOrders_SuperStoreOrders.csv')
]

data_path = None
for p in candidate_paths:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError('CSV not found. Update candidate_paths.')

df = pd.read_csv(data_path)
df.columns = [c.strip().replace(' ', '_') for c in df.columns]

for col in df.columns:
    if 'date' in col.lower():
        df[col] = pd.to_datetime(df[col], errors='coerce')

df = df.drop_duplicates().copy()

if 'Order_Date' in df.columns:
    df['Order_Year'] = df['Order_Date'].dt.year
    df['Order_Month'] = df['Order_Date'].dt.month

if {'Sales', 'Profit'}.issubset(df.columns):
    df['Profit_Margin'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'], np.nan)

if 'Discount' in df.columns:
    bins = [-0.01, 0, 0.1, 0.2, 0.4, 1]
    labels = ['No Discount', 'Low', 'Medium', 'High', 'Very High']
    df['Discount_Bucket'] = pd.cut(df['Discount'], bins=bins, labels=labels)

print('Loaded from:', data_path)
print('Shape:', df.shape)
display(df.head())

## 3) Strong Visual 1: Monthly Sales Trend

In [ ]:
if {'Order_Date', 'Sales'}.issubset(df.columns):
    monthly_sales = df.set_index('Order_Date').resample('M')['Sales'].sum().reset_index()
    plt.figure(figsize=(12, 4))
    sns.lineplot(data=monthly_sales, x='Order_Date', y='Sales', color='royalblue')
    plt.title('Monthly Sales Trend')
    plt.show()

## 4) Strong Visual 2: Monthly Profit Trend

In [ ]:
if {'Order_Date', 'Profit'}.issubset(df.columns):
    monthly_profit = df.set_index('Order_Date').resample('M')['Profit'].sum().reset_index()
    plt.figure(figsize=(12, 4))
    sns.lineplot(data=monthly_profit, x='Order_Date', y='Profit', color='forestgreen')
    plt.title('Monthly Profit Trend')
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.show()

## 5) Strong Visual 3: Sales by Category

In [ ]:
if {'Category', 'Sales'}.issubset(df.columns):
    cat_sales = df.groupby('Category', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
    plt.figure(figsize=(8, 4))
    sns.barplot(data=cat_sales, x='Category', y='Sales', palette='Blues_d')
    plt.title('Total Sales by Category')
    plt.show()

## 6) Strong Visual 4: Profit by Category

In [ ]:
if {'Category', 'Profit'}.issubset(df.columns):
    cat_profit = df.groupby('Category', as_index=False)['Profit'].sum().sort_values('Profit', ascending=False)
    plt.figure(figsize=(8, 4))
    sns.barplot(data=cat_profit, x='Category', y='Profit', palette='Greens_d')
    plt.title('Total Profit by Category')
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.show()

## 7) Strong Visual 5: Profit by Sub-Category (Worst to Best)

In [ ]:
if {'Sub_Category', 'Profit'}.issubset(df.columns):
    sub_profit = df.groupby('Sub_Category', as_index=False)['Profit'].sum().sort_values('Profit')
    plt.figure(figsize=(10, 7))
    sns.barplot(data=sub_profit, y='Sub_Category', x='Profit', palette='coolwarm')
    plt.title('Profit by Sub-Category')
    plt.axvline(0, color='black', linestyle='--', linewidth=1)
    plt.show()

## 8) Strong Visual 6: Region Performance (Sales and Profit)

In [ ]:
if {'Region', 'Sales', 'Profit'}.issubset(df.columns):
    region_perf = df.groupby('Region', as_index=False)[['Sales', 'Profit']].sum().sort_values('Sales', ascending=False)
    display(region_perf)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.barplot(data=region_perf, x='Region', y='Sales', ax=axes[0], palette='PuBu')
    axes[0].set_title('Sales by Region')

    sns.barplot(data=region_perf, x='Region', y='Profit', ax=axes[1], palette='YlGn')
    axes[1].set_title('Profit by Region')
    axes[1].axhline(0, color='red', linestyle='--', linewidth=1)

    plt.tight_layout()
    plt.show()

## 9) Strong Visual 7: Discount vs Profit

In [ ]:
if {'Discount', 'Profit'}.issubset(df.columns):
    sample_df = df.sample(min(5000, len(df)), random_state=42)
    plt.figure(figsize=(9, 5))
    sns.scatterplot(data=sample_df, x='Discount', y='Profit', alpha=0.4)
    plt.title('Discount vs Profit')
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.show()

## 10) Strong Visual 8: Average Profit by Discount Bucket

In [ ]:
if {'Discount_Bucket', 'Profit'}.issubset(df.columns):
    bucket_profit = df.groupby('Discount_Bucket', observed=False)['Profit'].mean().reset_index()
    plt.figure(figsize=(9, 4))
    sns.barplot(data=bucket_profit, x='Discount_Bucket', y='Profit', palette='Spectral')
    plt.title('Average Profit by Discount Bucket')
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.show()
    display(bucket_profit)

## 11) Machine Learning Snapshot (Predict Profit)

This section gives one clean, interview-ready regression pipeline.

In [ ]:
if 'Profit' not in df.columns:
    raise ValueError('Profit column required for ML section.')

ml_df = df.copy()
drop_cols = ['Profit']
for c in ['Row_ID', 'Order_ID', 'Customer_ID', 'Customer_Name', 'Product_Name']:
    if c in ml_df.columns:
        drop_cols.append(c)

X = ml_df.drop(columns=drop_cols, errors='ignore')
y = ml_df['Profit']

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
}

ml_results = []
pred_store = {}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    ml_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred)
    })
    pred_store[name] = pred

ml_results_df = pd.DataFrame(ml_results).sort_values('R2', ascending=False)
display(ml_results_df)

best_name = ml_results_df.iloc[0]['Model']
best_pred = pred_store[best_name]

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_pred, alpha=0.4)
low = min(y_test.min(), best_pred.min())
high = max(y_test.max(), best_pred.max())
plt.plot([low, high], [low, high], 'r--')
plt.title('Actual vs Predicted Profit: ' + best_name)
plt.xlabel('Actual Profit')
plt.ylabel('Predicted Profit')
plt.show()

## 12) Five Key Insights (Fill After Running)

Use this exact format in interviews:
1. The strongest revenue driver is ______.
2. The strongest profit driver is ______.
3. The largest profit leak is in ______.
4. High discount tiers show ______ impact on average profit.
5. Best ML model for profit prediction is ______ with R2 = ______.

## 13) Three Business Recommendations

1. Build discount guardrails for products/sub-categories with repeated losses.
2. Focus inventory and marketing on high-profit segments and regions.
3. Use ML score as an order-risk signal before pricing finalization.

## 14) Executive Conclusion

This analysis identifies where Superstore earns and where it loses profit.
The visuals reveal business patterns, and the ML model provides predictive support.
Together, they enable data-driven pricing, product focus, and profitability improvement.